# 01 — Data Cleaning & Preprocessing
### PM Accelerator — Weather Trend Forecasting Assessment

This notebook:
- Loads the raw Global Weather Repository CSV
- Parses `last_updated` as datetime and sorts ascending
- Logs and imputes missing values (median for numeric columns)
- Flags outliers via IQR on key numeric columns
- Normalizes numeric features with StandardScaler (saved for reuse)
- Saves the cleaned DataFrame to `data/cleaned_weather.csv`

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from src.utils import load_raw_data, save_scaler, DATA_DIR, MODELS_DIR

## 1. Load Raw Data

In [2]:
df = load_raw_data()
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
print(f'Date range: {df["last_updated"].min()} → {df["last_updated"].max()}')
df.head()

Shape: (130783, 41)
Columns (41): ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']
Date range: 2024-05-16 01:45:00 → 2026-03-21 19:30:00


c:\Users\purva\OneDrive\Desktop\PMA\notebooks\..\src\utils.py:33: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["last_updated"] = pd.to_datetime(df["last_updated"], infer_datetime_format=True)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15:00,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45:00,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45:00,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45:00,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45:00,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


## 2. Set Datetime Index (Sorted Ascending)

In [3]:
df.sort_values('last_updated', inplace=True)
df.set_index('last_updated', inplace=True)
print(f'Index dtype: {df.index.dtype}')
print(f'First 5 dates:\n{df.index[:5].tolist()}')
df.head()

Index dtype: datetime64[ns]
First 5 dates:
[Timestamp('2024-05-16 01:45:00'), Timestamp('2024-05-16 02:45:00'), Timestamp('2024-05-16 02:45:00'), Timestamp('2024-05-16 02:45:00'), Timestamp('2024-05-16 02:45:00')]


,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
last_updated,,,,,,,,,,,,,,,,,,,,,
2024-05-16 01:45:00,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,16.1,61.0,Clear,4.3,...,6.3,7.1,1,1,05:26 AM,08:31 PM,01:36 PM,02:52 AM,Waxing Gibbous,55
2024-05-16 02:45:00,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,26.0,78.8,Moderate or heavy rain with thunder,2.2,...,20.4,28.1,2,2,05:30 AM,06:16 PM,01:00 PM,01:02 AM,Waxing Gibbous,55
2024-05-16 02:45:00,Costa Rica,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,21.0,69.8,Fog,2.2,...,21.7,23.3,2,2,05:15 AM,05:51 PM,12:42 PM,12:37 AM,Waxing Gibbous,55
2024-05-16 02:45:00,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,20.0,68.0,Mist,13.6,...,132.0,178.1,4,10,05:34 AM,06:23 PM,01:05 PM,01:09 AM,Waxing Gibbous,55
2024-05-16 02:45:00,Nicaragua,Managua,12.15,-86.27,America/Managua,1715849100,27.2,80.9,Patchy rain nearby,3.6,...,11.7,14.7,1,1,05:21 AM,06:02 PM,12:49 PM,12:49 AM,Waxing Gibbous,55


## 3. Missing Value Analysis & Imputation

In [4]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
missing_report = missing_report[missing_report['null_count'] > 0].sort_values('null_count', ascending=False)

if len(missing_report) > 0:
    print('Columns with missing values:')
    display(missing_report)
else:
    print('No missing values found.')

No missing values found.


In [5]:
# Drop rows where more than 50% of values are null
row_null_pct = df.isnull().mean(axis=1)
rows_to_drop = row_null_pct[row_null_pct > 0.5]
if len(rows_to_drop) > 0:
    print(f'Dropping {len(rows_to_drop)} rows with >50% nulls')
    df = df.loc[row_null_pct <= 0.5]

# Impute numeric columns with median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Imputing {len(numeric_cols)} numeric columns with column median...')
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        n_imputed = df[col].isnull().sum()
        df[col].fillna(median_val, inplace=True)
        print(f'  {col}: imputed {n_imputed} values with median={median_val:.2f}')

remaining_nulls = df[numeric_cols].isnull().sum().sum()
print(f'\nRemaining nulls in numeric columns: {remaining_nulls}')

Imputing 30 numeric columns with column median...

Remaining nulls in numeric columns: 0


## 4. IQR Outlier Flagging

Flag outliers using the Interquartile Range method on key numeric columns. **Outliers are NOT dropped** — they are kept for use in the anomaly detection notebook.

In [6]:
outlier_cols = ['temperature_celsius', 'precip_mm', 'wind_mph']
df['is_outlier'] = False

for col in outlier_cols:
    if col not in df.columns:
        print(f'  Warning: Column "{col}" not found, skipping')
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df[col] < lower) | (df[col] > upper)
    count = mask.sum()
    df.loc[mask, 'is_outlier'] = True
    print(f'  {col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, '
          f'bounds=[{lower:.2f}, {upper:.2f}], outliers={count}')

total_outlier_rows = df['is_outlier'].sum()
print(f'\nTotal rows flagged as outlier: {total_outlier_rows} '
      f'({total_outlier_rows/len(df)*100:.2f}%)')
print('NOTE: Outliers are NOT dropped — they are kept for anomaly detection.')

  temperature_celsius: Q1=16.10, Q3=28.00, IQR=11.90, bounds=[-1.75, 45.85], outliers=2697
  precip_mm: Q1=0.00, Q3=0.03, IQR=0.03, bounds=[-0.04, 0.07], outliers=24300
  wind_mph: Q1=3.80, Q3=11.00, IQR=7.20, bounds=[-7.00, 21.80], outliers=2096

Total rows flagged as outlier: 28134 (21.51%)
NOTE: Outliers are NOT dropped — they are kept for anomaly detection.


## 5. Feature Normalization (StandardScaler)

In [7]:
exclude_from_scaling = ['is_outlier', 'latitude', 'longitude',
                        'last_updated_epoch', 'moon_illumination']
scale_cols = [c for c in numeric_cols if c not in exclude_from_scaling]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])
save_scaler(scaler, 'standard_scaler')
print(f'Scaled {len(scale_cols)} columns.')

Scaler saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\models\standard_scaler.pkl
Scaled 26 columns.


## 6. Save Cleaned Data

In [8]:
# Save scaled version
output_path = os.path.join(DATA_DIR, 'cleaned_weather.csv')
df.to_csv(output_path)
print(f'Saved cleaned DataFrame → {output_path}')
print(f'Final shape: {df.shape}')

# Also save an UNSCALED version for EDA and modelling
print('\nAlso saving unscaled version for EDA/modelling...')
df_unscaled = load_raw_data()
df_unscaled.sort_values('last_updated', inplace=True)
df_unscaled.set_index('last_updated', inplace=True)

# Reapply imputation
for col in numeric_cols:
    if col in df_unscaled.columns and df_unscaled[col].isnull().any():
        df_unscaled[col].fillna(df_unscaled[col].median(), inplace=True)

# Reapply outlier flag
df_unscaled['is_outlier'] = False
for col in outlier_cols:
    if col not in df_unscaled.columns:
        continue
    Q1 = df_unscaled[col].quantile(0.25)
    Q3 = df_unscaled[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df_unscaled[col] < lower) | (df_unscaled[col] > upper)
    df_unscaled.loc[mask, 'is_outlier'] = True

unscaled_path = os.path.join(DATA_DIR, 'cleaned_weather_unscaled.csv')
df_unscaled.to_csv(unscaled_path)
print(f'Saved unscaled DataFrame → {unscaled_path}')

print('\n✅ Data cleaning complete!')

Saved cleaned DataFrame → c:\Users\purva\OneDrive\Desktop\PMA\data\cleaned_weather.csv
Final shape: (130783, 41)

Also saving unscaled version for EDA/modelling...


c:\Users\purva\OneDrive\Desktop\PMA\notebooks\..\src\utils.py:33: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["last_updated"] = pd.to_datetime(df["last_updated"], infer_datetime_format=True)


Saved unscaled DataFrame → c:\Users\purva\OneDrive\Desktop\PMA\data\cleaned_weather_unscaled.csv

✅ Data cleaning complete!
